
# 04 — Model Experiments

## 1. Objectives

We will experiment with different TF-IDF configurations
to improve the classification performance.

Current results:

| Model | Accuracy | Macro F1 |
|---|---|---|
| Baseline | 0.7701 | 0.5708 |
| Balanced Logistic Regression | 0.7877 | 0.6984 |

## Our Approach

1. Keep the same dataset splits.
2. Change one parameter at a time.
3. Evaluate using validation data.
4. Use Macro F1 as the primary metric.
5. Keep the test set untouched.



## 2. Load Data and Reproduce Splits

Load the cleaned dataset and reproduce the same stratified
train/validation/test split used in the baseline notebook.

The test set must remain untouched during model selection.


In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/processed/cleaned_complaints.csv")

TEXT_COL = "Consumer complaint narrative"
TARGET_COL = "Product"

X = df[TEXT_COL]
y = df[TARGET_COL]

# First split: 80% train+validation, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

# Second split: 64% train, 16% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.20,
    stratify=y_train_val,
    random_state=42
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))
print("Total:", len(X_train) + len(X_val) + len(X_test))



## 3. Experiment 1 — Minimum Document Frequency

Compare min_df=1 and min_df=2 while keeping:

- TF-IDF ngram_range=(1, 2)
- Logistic Regression class_weight="balanced"
- max_iter=1000

Primary metric: Validation Macro F1.

Only one hyperparameter changes between experiments.


In [ ]:

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

model_min1 = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=1,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model_min1.fit(X_train, y_train)

print("Model with min_df=1 trained successfully!")



## 4. Experiment 1 — Validation Results

Evaluate the model with min_df=1 and compare it
against the previous balanced baseline (min_df=2).

Primary metric: Macro F1.


In [ ]:

y_pred_min1 = model_min1.predict(X_val)

accuracy_min1 = accuracy_score(y_val, y_pred_min1)

macro_f1_min1 = f1_score(
    y_val,
    y_pred_min1,
    average="macro"
)

print(f"min_df=1 Accuracy: {accuracy_min1:.4f}")
print(f"min_df=1 Macro F1: {macro_f1_min1:.4f}")

print("\nPrevious Balanced Model (min_df=2):")
print("Accuracy: 0.7877")
print("Macro F1: 0.6984")



## 5. Experiment 2 — Unigrams vs Bigrams

Compare unigram-only features against unigram + bigram features.

Keep all other model parameters unchanged.

Primary metric: Validation Macro F1.


In [ ]:

model_unigrams = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=2,
        ngram_range=(1, 1)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model_unigrams.fit(X_train, y_train)

print("Unigram-only model trained successfully!")



## 6. Experiment 2 — Validation Results

Compare unigram-only TF-IDF with unigram + bigram TF-IDF.

Evaluate using Accuracy and Macro F1 on the same validation set.


In [ ]:

y_pred_unigrams = model_unigrams.predict(X_val)

accuracy_unigrams = accuracy_score(
    y_val, y_pred_unigrams
)

macro_f1_unigrams = f1_score(
    y_val,
    y_pred_unigrams,
    average="macro"
)

print(f"Unigrams Accuracy: {accuracy_unigrams:.4f}")
print(f"Unigrams Macro F1: {macro_f1_unigrams:.4f}")

print("\nUnigrams + Bigrams (Previous Best):")
print("Accuracy: 0.7877")
print("Macro F1: 0.6984")



## 7. Vocabulary Size Comparison

Compare the number of TF-IDF features learned by:

- Unigrams only
- Unigrams + Bigrams

Analyze the trade-off between model complexity and performance.


In [ ]:

unigram_features = len(
    model_unigrams.named_steps["tfidf"].get_feature_names_out()
)

bigram_features = len(
    model_min1.named_steps["tfidf"].get_feature_names_out()
)

print("Unigram features:", unigram_features)
print("min_df=1 Unigrams + Bigrams:", bigram_features)


In [ ]:

model_unigrams_bigrams = Pipeline([
    ("tfidf", TfidfVectorizer(
        min_df=2,
        ngram_range=(1, 2)
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

model_unigrams_bigrams.fit(X_train, y_train)

unigram_features = len(
    model_unigrams.named_steps["tfidf"].get_feature_names_out()
)

unigram_bigram_features = len(
    model_unigrams_bigrams.named_steps["tfidf"].get_feature_names_out()
)

print("Unigrams only:", unigram_features)
print("Unigrams + Bigrams:", unigram_bigram_features)
print("Additional features:", unigram_bigram_features - unigram_features)



## 8. Experiment Tracking

Record the hyperparameters and validation metrics
for each experiment.

Macro F1 is the primary model-selection metric.
The test set remains untouched.


In [ ]:

experiments = pd.DataFrame([
    {
        "Model": "Baseline",
        "min_df": 2,
        "ngram_range": "(1, 2)",
        "class_weight": "None",
        "Accuracy": 0.7701,
        "Macro F1": 0.5708
    },
    {
        "Model": "Balanced",
        "min_df": 2,
        "ngram_range": "(1, 2)",
        "class_weight": "balanced",
        "Accuracy": 0.7877,
        "Macro F1": 0.6984
    },
    {
        "Model": "Balanced min_df=1",
        "min_df": 1,
        "ngram_range": "(1, 2)",
        "class_weight": "balanced",
        "Accuracy": accuracy_min1,
        "Macro F1": macro_f1_min1
    },
    {
        "Model": "Balanced unigrams",
        "min_df": 2,
        "ngram_range": "(1, 1)",
        "class_weight": "balanced",
        "Accuracy": accuracy_unigrams,
        "Macro F1": macro_f1_unigrams
    }
])

experiments.sort_values(
    by="Macro F1",
    ascending=False
).round(4)



## 9. Stratified Cross-Validation

We evaluate the current best configuration using
3-fold stratified cross-validation on the training set.

Configuration:
- TF-IDF: min_df=2, ngram_range=(1, 2)
- Logistic Regression: class_weight="balanced"
- Primary metric: Macro F1

The held-out validation and test sets are not used.


In [ ]:

from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    model_unigrams_bigrams,
    X_train,
    y_train,
    cv=cv,
    scoring="f1_macro",
    n_jobs=1
)

print("F1 scores for each fold:", cv_scores)
print(f"Mean Macro F1: {cv_scores.mean():.4f}")
print(f"Standard deviation: {cv_scores.std():.4f}")



## 10. GridSearchCV — Regularization Strength

Use 3-fold stratified cross-validation to compare
different regularization strengths for Logistic Regression.

Keep the TF-IDF configuration fixed.

Primary metric: Macro F1.


In [ ]:

from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.5, 1.0, 2.0]
}

grid_search = GridSearchCV(
    estimator=model_unigrams_bigrams,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    refit=True
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best CV Macro F1: {grid_search.best_score_:.4f}")



## 11. GridSearch Model Validation

Evaluate the best hyperparameter configuration selected
by GridSearchCV on the held-out validation set.

Compare its performance with the previous balanced model.

The test set remains untouched.


In [ ]:

best_grid_model = grid_search.best_estimator_

y_pred_grid = best_grid_model.predict(X_val)

accuracy_grid = accuracy_score(y_val, y_pred_grid)

macro_f1_grid = f1_score(
    y_val,
    y_pred_grid,
    average="macro"
)

print(f"GridSearch Validation Accuracy: {accuracy_grid:.4f}")
print(f"GridSearch Validation Macro F1: {macro_f1_grid:.4f}")

print("\nPrevious Balanced Model (C=1.0):")
print("Accuracy: 0.7877")
print("Macro F1: 0.6984")



## 12. Tuned Model — Per-Class Evaluation

Analyze the best GridSearch model on the validation set.

Check whether hyperparameter tuning improves minority classes
or mainly improves the majority classes.

The test set remains untouched.


In [ ]:

from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_pred_grid,
    zero_division=0
))



## 13. Error Analysis — Debt or Credit Management

Investigate how the tuned model misclassifies complaints
belonging to the Debt or credit management category.

Use validation predictions only.


In [ ]:

target_class = "Debt or credit management"

error_analysis = pd.DataFrame({
    "Actual": y_val.to_numpy(),
    "Predicted": y_pred_grid
})

target_errors = error_analysis[
    (error_analysis["Actual"] == target_class) &
    (error_analysis["Predicted"] != target_class)
]

print("Misclassified complaints:", len(target_errors))

print("\nPredicted categories:")
print(target_errors["Predicted"].value_counts())



## 14. Inspect Misclassified Examples

Inspect short, redacted excerpts from validation complaints
to investigate possible feature overlap.

Focus on Debt or credit management complaints
misclassified as Debt collection.


In [ ]:

import re

error_examples = error_analysis.copy()

error_examples["Text"] = X_val.to_numpy()

debt_errors = error_examples[
    (error_examples["Actual"] == "Debt or credit management") &
    (error_examples["Predicted"] == "Debt collection")
].copy()

def safe_preview(text):
    text = re.sub(
        r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b",
        "[EMAIL]",
        text
    )
    text = re.sub(r"\b\d{4,}\b", "[NUMBER]", text)
    return text[:200]

debt_errors["Preview"] = debt_errors["Text"].apply(safe_preview)

debt_errors[["Actual", "Predicted", "Preview"]].head(3)



## 15. Error Analysis — Keyword Patterns

Investigate whether misclassified debt management complaints
contain terminology associated with debt collection or
debt management.

This is exploratory analysis, not proof of causation.


In [ ]:

keywords = [
    "collection",
    "collector",
    "settlement",
    "counseling",
    "repayment",
    "management",
    "payment",
    "debt"
]

for keyword in keywords:
    count = debt_errors["Text"].str.contains(
        rf"\b{keyword}\b",
        case=False,
        regex=True,
        na=False
    ).sum()

    print(f"{keyword}: {count} complaints")



## 16. Correct vs Incorrect — Keyword Comparison

Compare keyword prevalence among correctly and incorrectly
classified Debt or credit management complaints.

Keyword prevalence measures the percentage of complaints
containing a keyword at least once.


In [ ]:

target_samples = error_examples[
    error_examples["Actual"] == target_class
].copy()

target_samples["Correct"] = (
    target_samples["Actual"] == target_samples["Predicted"]
)

keywords = [
    "debt", "collection", "collector",
    "settlement", "counseling", "repayment",
    "management", "payment"
]

results = []

for keyword in keywords:
    contains_word = target_samples["Text"].str.contains(
        rf"\b{keyword}\b",
        case=False,
        regex=True,
        na=False
    )

    results.append({
        "Keyword": keyword,
        "Correct (%)": (
            contains_word[target_samples["Correct"]].mean() * 100
        ),
        "Incorrect (%)": (
            contains_word[~target_samples["Correct"]].mean() * 100
        )
    })

keyword_comparison = pd.DataFrame(results)

keyword_comparison.round(1)
